# MMJL Capture and Notebook-State Regression Test

**1. Markdown cell — test purpose and execution order**

This notebook tests:

1. repository-root discovery from anywhere inside the repository
2. `src/` import setup
3. MMJL utility imports
4. Jupyter magic registration
5. literal Markdown and HTML logging
6. `%%jupy_capture`
7. the `%%jupy_tee` alias
8. failed notebook-state execution in visible order:
   `A -> B -> C`
9. successful recovery by moving backward:
   `B -> A`
10. stdout, exception, rich-display, and Matplotlib capture
11. `%jupy_file`
12. manifest inspection and validation
13. Markdown and HTML timeline generation
14. the Python analogs of `tree`, `wc -l`, and `cat`

The notebook-state dependency graph is:

```text
C creates numbers
B consumes numbers and creates squared_numbers
A consumes numbers and squared_numbers, computes a mean, and plots
```

The cells are displayed as:

```text
A
B
C
B again
A again
```

Therefore the first A and B should fail, C should succeed, and the
subsequent B and A should succeed.

---

**Looking further**:

Here is the complete notebook sequence. The intended execution order is the notebook's ordinary top-to-bottom order, with the exception of the A → B → C → B → A example

---

**Let's go!!!**<br/><br/>

In [ ]:
## 2. Code cell 01 — locate the repository root and configure `sys.path`
import importlib
import os
import sys
import pathlib

starting_dir = pathlib.Path.cwd().resolve()
repo_root = None
src_path = None
package_path = None

for candidate_path in [starting_dir, *starting_dir.parents]:
    package_path = (
        candidate_path
        / "src"
        / "multimodal_jupy_logger"
    )

    if package_path.is_dir():
        repo_root = candidate_path
        break
    ##endof:  if package_path.is_dir()
##endof:  for candidate_path in [...]

if repo_root is None:
    raise RuntimeError(
        "Could not find a repository root containing "
        "src/multimodal_jupy_logger."
    )
##endof:  if repo_root is None

src_path = repo_root / "src"

os.chdir(repo_root)

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
##endof:  if str(src_path) not in sys.path

importlib.invalidate_caches()

print("starting_dir:", starting_dir)
print("repo_root:", repo_root)
print("src_path:", src_path)
print("current working directory:", pathlib.Path.cwd())
print("Python executable:", sys.executable)

In [ ]:
## 3. Code cell 02 — import MMJL and the path-display utilities
from multimodal_jupy_logger import (
    MultimodalJupyLogger,
    jupy_logger_register,
    register_jupy_logger,
)

from multimodal_jupy_logger.utils import (
    count_lines,
    path_display,
    print_file,
    tree,
)

from multimodal_jupy_logger.utils import path_display as dpypd

print("MMJL imports succeeded.")
print("path_display module:", dpypd)

## Manifest and Artifact Lookup Helpers

This notebook is a development lab, so this cell gives us stable ways to
inspect MMJL output after a kernel restart. The manifest and artifacts live
on disk, even when Python state is gone.

Use examples:

```python
show_manifest()
show_manifest(label_contains="abc-second-pass-a")
show_artifact(23)
```


In [ ]:
## Code cell 03

## Development helper cell -- inspect manifest rows and artifacts.
## This cell is notebook-local; it is not part of the MMJL public API.

import csv
import json
import mimetypes
import pathlib

from typing import Any

from IPython.display import HTML, Image, Markdown, display


ManifestRow = dict[str, Any]


def find_repo_root(
      starting_dir: pathlib.Path | str | None = None,
    ) -> pathlib.Path:
    '''
    Find the repository root from the current notebook location.
    '''

    starting_path = (
        pathlib.Path.cwd()
        if starting_dir is None
        else pathlib.Path(starting_dir)
    )
    starting_path = starting_path.resolve()

    for candidate_path in [starting_path, *starting_path.parents]:
        package_path = (
            candidate_path
            / "src"
            / "multimodal_jupy_logger"
        )

        if package_path.is_dir():
            return candidate_path
        ##endof:  if package_path.is_dir()
    ##endof:  for candidate_path in [...]

    raise RuntimeError("Could not find MMJL repository root.")
##endof:  find_repo_root


def read_mmjl_manifest(
      log_root: pathlib.Path | str | None = None,
    ) -> tuple[pathlib.Path, list[ManifestRow]]:
    '''
    Read the MMJL manifest as TSV rows.
    '''

    if log_root is None:
        local_repo_root = find_repo_root()
        log_root = local_repo_root / "jupy_log"
    ##endof:  if log_root is None

    log_root = pathlib.Path(log_root)
    manifest_path = log_root / "manifest.tsv"

    with manifest_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle, delimiter="\t"))
    ##endof:  with manifest_path.open(...)

    return manifest_path, rows
##endof:  read_mmjl_manifest


def _row_int(
      row: ManifestRow,
      key: str,
      default: int = -1,
    ) -> int:
    '''
    Read an integer-ish manifest field for sorting/filtering.
    '''

    value = row.get(key, "")

    try:
        return int(value)
    except (TypeError, ValueError):
        return default
    ##endof:  try/except (TypeError, ValueError)
##endof:  _row_int


def show_manifest(
      limit: int = 20,
      log_root: pathlib.Path | str | None = None,
      label_contains: str | None = None,
    ) -> None:
    '''
    Print a compact view of recent manifest rows.
    '''

    manifest_path, rows = read_mmjl_manifest(log_root=log_root)

    if label_contains is not None:
        rows = [
            row for row in rows
            if label_contains.lower() in row.get("label", "").lower()
        ]
    ##endof:  if label_contains is not None

    rows = sorted(
        rows,
        key=lambda row: _row_int(row, "artifact_sequence"),
    )

    print(f"manifest: {manifest_path}")
    print(f"rows: {len(rows)}")
    print()

    for row in rows[-limit:]:
        print(
            row.get("artifact_sequence", ""),
            row.get("capture_sequence", ""),
            row.get("role", ""),
            row.get("mime", ""),
            row.get("label", ""),
            row.get("path", ""),
            sep=" | ",
        )
    ##endof:  for row in rows[-limit:]
##endof:  show_manifest


def get_artifact_row(
      artifact_sequence: int,
      log_root: pathlib.Path | str | None = None,
    ) -> tuple[pathlib.Path, ManifestRow]:
    '''
    Return the manifest row for one artifact sequence.
    '''

    manifest_path, rows = read_mmjl_manifest(log_root=log_root)
    target = int(artifact_sequence)

    for row in rows:
        if _row_int(row, "artifact_sequence") == target:
            return manifest_path, row
        ##endof:  if _row_int(row, "artifact_sequence") == target
    ##endof:  for row in rows

    raise KeyError(f"No artifact_sequence found: {artifact_sequence}")
##endof:  get_artifact_row


def resolve_artifact_path(
      manifest_path: pathlib.Path | str,
      row: ManifestRow,
    ) -> pathlib.Path:
    '''
    Resolve a manifest artifact path against the manifest directory.
    '''

    manifest_dir = pathlib.Path(manifest_path).parent
    artifact_path = pathlib.Path(row.get("path", ""))

    if not artifact_path.is_absolute():
        artifact_path = manifest_dir / artifact_path
    ##endof:  if not artifact_path.is_absolute()

    return artifact_path.resolve()
##endof:  resolve_artifact_path


def show_artifact(
      artifact_sequence: int,
      log_root: pathlib.Path | str | None = None,
    ) -> ManifestRow:
    '''
    Display one logged artifact by its monotonic artifact sequence.
    '''

    manifest_path, row = get_artifact_row(
        artifact_sequence,
        log_root=log_root,
    )
    artifact_path = resolve_artifact_path(manifest_path, row)
    guessed_mime = mimetypes.guess_type(artifact_path)[0]
    mime = row.get("mime", "") or guessed_mime or ""

    print(f"artifact_sequence: {row.get('artifact_sequence')}")
    print(f"capture_sequence: {row.get('capture_sequence')}")
    print(f"role: {row.get('role')}")
    print(f"label: {row.get('label')}")
    print(f"mime: {mime}")
    print(f"path: {artifact_path}")
    print(f"exists: {artifact_path.exists()}")
    print()

    if not artifact_path.exists():
        return row
    ##endof:  if not artifact_path.exists()

    if mime == "image/png":
        display(Image(filename=str(artifact_path)))
    elif mime == "image/svg+xml":
        display(HTML(artifact_path.read_text(encoding="utf-8")))
    elif mime in ["text/markdown", "text/x-markdown"]:
        display(Markdown(artifact_path.read_text(encoding="utf-8")))
    elif mime == "text/html":
        display(HTML(artifact_path.read_text(encoding="utf-8")))
    elif mime == "application/json":
        print(json.dumps(
            json.loads(artifact_path.read_text(encoding="utf-8")),
            indent=2,
        ))
    elif mime.startswith("text/"):
        print(artifact_path.read_text(
            encoding="utf-8",
            errors="replace",
        ))
    else:
        print("No display rule for this MIME type.")
    ##endof:  if/elif MIME display

    return row
##endof:  show_artifact


def display_banner(my_char: str='-', my_len: int=65) -> None: 
    print(my_len*my_char)
##endof:  display_banner(...)

In [ ]:
## 4. Code cell 04 — prove both utility import styles work

print("Directly imported function:")
print("  tree:", tree)

print("\nModule-namespace functions:")
print("  dpypd.tree:", dpypd.tree)
print("  dpypd.count_lines:", dpypd.count_lines)
print("  dpypd.print_file:", dpypd.print_file)

print("\nDirect and module attributes refer to the same functions:")
print("  tree is dpypd.tree:", tree is dpypd.tree)
print(
    "  count_lines is dpypd.count_lines:",
    count_lines is dpypd.count_lines,
)
print(
    "  print_file is dpypd.print_file:",
    print_file is dpypd.print_file,
)

In [ ]:
## 5. Code cell 05 — show the relevant repository tree

dpypd.tree(
    this_dir=repo_root,
    dirs_to_exclude=[
        ".git",
        "__pycache__",
        ".venv",
        ".venv_test_mmjl",
        ".ipynb_checkpoints",
    ],
    files_to_exclude=[
        ".pyc",
    ],
)

For:  **6**

Run this after restarting the kernel and loading the updated files. 
(If you've run the last cells, this is almost certainly done.)

```python
register_jupy_logger()
```

Expected registration text includes:

```text
%%jupy_log
%%jupy_capture
%%jupy_tee
```

In [ ]:
## 6. Code cell 06 — register the magics

register_jupy_logger()

In [ ]:
## 7. Code cell 07 — verify that the magics exist

ipython_shell = get_ipython()

magic_names = [
    "jupy_save",
    "jupy_file",
    "jupy_log",
    "jupy_capture",
    "jupy_tee",
    "jupy_markdown",
    "jupy_html",
    "jupy_inspect",
    "jupy_validate",
]

for magic_name in magic_names:
    line_magic = ipython_shell.find_line_magic(magic_name)
    cell_magic = ipython_shell.find_cell_magic(magic_name)

    print(
        f"{magic_name:16s}",
        f"line={line_magic is not None}",
        f"cell={cell_magic is not None}",
    )
##endof:  for magic_name in magic_names

**8. Markdown cell — notebook-native hidden-text control**

This cell tests Jupyter’s own Markdown rendering independently of MMJL.

```markdown
<strong>Notebook-native `<details>` control</strong>

<details>
<summary>Click the arrow to reveal the notebook-native text</summary>

This text lives directly in a Jupyter Markdown cell.

If clicking the arrow reveals this paragraph, the notebook frontend is
rendering the HTML element correctly.

</details>
```

---

_DWB Note: I think we want to test the following:_

<strong>Notebook-native `<details>` control</strong>

<details>
<summary>Click the arrow to reveal the notebook-native text</summary>

This text lives directly in a Jupyter Markdown cell.

If clicking the arrow reveals this paragraph, the notebook frontend is
rendering the HTML element correctly.

</details>

<br/><hr/>

For **9** (`Code cell 08 — log the original Markdown regression case`)

This preserves the original test: raw `<details>` HTML stored with the
`text/markdown` MIME type.

In [ ]:
%%jupy_log --label details-summary-markdown-regression --mime text/markdown
<details>
<summary>Click the arrow to reveal logged Markdown text</summary>

This content was logged as `text/markdown`.

The regression question is whether exported timelines preserve this as
renderable Markdown/HTML or incorrectly turn it into escaped or fenced
source text.

</details>

<!-- ## 9. Code cell 08 — log the original Markdown regression case -->
<!--   If I had left the python comment without the HTML comment,   -->
<!-- + it would have rendered as an `h2`. But you might only see    -->
<!-- + this in the dpypd.print_file version.                        -->

<br/>HTML comments above.<br/>Hopefully you can see the utility of jupy_log
as a note-taking device.

For **10** (`Code cell 09 — log an HTML control version`)

This distinguishes a general `<details>` failure from a Markdown-MIME
rendering-policy failure.

With the current timeline builder, the expected distinction is:

* `text/html`: should render as a collapsible block in the HTML timeline
* `text/markdown`: currently may appear as escaped/fenced source

That would identify a timeline-rendering issue, not a capture failure.

In [ ]:
%%jupy_log --label details-summary-html-control --mime text/html
<details>
<summary>Click the arrow to reveal logged HTML text</summary>

<p>
This content was logged as <code>text/html</code>. It should be inserted
as HTML in the generated HTML timeline.
</p>

</details>

## 10. Code cell 09 — log an HTML control version

<!--   I can leave the python comment above as-is, since this is -->
<!-- + HTML and not markdown. But you might only see this in the -->
<!-- + dpypd.print_file version.                                 -->

<br/>HTML comments above.<br/>Hopefully you can see the utility of jupy_log
as a note-taking device.

In [ ]:
# Code cell 10

dpypd.tree(
    this_dir=repo_root,
    dirs_to_exclude=[
        ".git",
        "__pycache__",
        ".venv",
        ".venv_test_mmjl",
        ".ipynb_checkpoints",
    ],
    files_to_exclude=[
        ".pyc",
    ],
)

In [ ]:
## Code cell 11

dpypd.tree(f"{repo_root}/jupy_log")

In [ ]:
## Code cell 12, better manifest display

display_banner()
show_manifest(limit=20)
display_banner()
print()
print("That's the previous tries and the start of this one...")
print("Unless, as is the case now, this is a different machine,")
print("so there are only the two new entries.")

<br/><hr/>
<div>
<span style="font-size:150%; font-face:bold">MUST 
    CHANGE (or at least check) <code>show_artifact</code> NUMBERS 
    BELOW!</span>
</div>
<br/>
<div>
<span style="font-size:200%; font-face:bold">ABSOLUTELY MUST
    &nbsp;</span><span style="font-size:150%; font-face:bold">MANUALLY
    PUT IN THE FIRST PART OF THE FILENAMES, BELOW, FROM THE FIRST
    NUMBER THROUGH THE LAST UNDERSCORE!</span>
</div>
<div>(The <code>path_second_part</code> variables,
    of which there are two.)</div>

In [ ]:
##  Code cell 13 — ANYTHING NEW?

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(1)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(2)
display_banner()
print(); print()
display_banner("#", 72)

print();print();print()
display_banner("#", 68)
print("### ONLY FOR THIS REGRESSION TEST, WE dpypd.print_file TWO FILES ###")
display_banner("#", 68)
print();print();print()

##  Refer to src = "bballdave025/multimodal_jupy_logger/" + 
##+   "/mmjl_capture_and_state_smoke_test_1781817012.ipynb"
##+ for how to use this compact code.
hashes="#"*63; dashes="-"*59; equalses="="*59; print_start="___Fil" + \
"e contents for___\n   'jupy_log/"; count_start="___File line coun" + \
"t for___\n   'jupy_log/"; full_stem_path=f"{repo_root}/jupy_log/";
either_end=f"'___\n{dashes}\n";     print(f"{hashes}\n{dashes}\n");

the_right_start=the_right_end="!";dpath_post_jupy_log="";art_str="";
path_second_part="";path_third_part="";
the_right_start = print_start;
#a.2  #the_right_start = count_start;
art_str="artifacts/";
short_pth="";
dpath_post_jupy_log=f"{art_str}{short_pth}";
path_second_part="000001";
path_third_part ="details-summary-markdown-regression.md";
dpath_post_jupy_log+=f"\n{path_second_part}\n{path_third_part}";
path_pass=(  f"{full_stem_path}{art_str}{short_pth}{path_second_part}"
f"{path_third_part}");print((f"{the_right_start}{dpath_post_jupy_log}"
f"{either_end}") ); dpypd.print_file(path_pass) \
if the_right_start == print_start  else dpypd.count_lines(path_pass); 
contd_bigger_end=f"{dashes}\n{equalses}\n\n{equalses}\n{dashes}\n";
ended_bigger_end=f"{dashes}\n\n\n{hashes}";
the_right_end=contd_bigger_end;
#d.2  #the_right_end=ended_bigger_end;
print(the_right_end)

the_right_start=the_right_end="!";dpath_post_jupy_log="";art_str="";
path_second_part="";path_third_part="";
the_right_start = print_start;
#a.2  #the_right_start = count_start;
art_str="artifacts/";
short_pth="";
dpath_post_jupy_log=f"{art_str}{short_pth}";
path_second_part="000002";
path_third_part ="details-summary-html-control.html";
dpath_post_jupy_log+=f"\n{path_second_part}\n{path_third_part}";
path_pass=(  f"{full_stem_path}{art_str}{short_pth}{path_second_part}"
f"{path_third_part}");print((f"{the_right_start}{dpath_post_jupy_log}"
f"{either_end}") ); dpypd.print_file(path_pass) \
if the_right_start == print_start  else dpypd.count_lines(path_pass); 
contd_bigger_end=f"{dashes}\n{equalses}\n\n{equalses}\n{dashes}\n";
ended_bigger_end=f"{dashes}\n\n\n{hashes}";
#d.1  #the_right_end=contd_bigger_end;
the_right_end=ended_bigger_end;
print(the_right_end)

---

**New stuff after Code cell 13**

<details>
<summary>Click the right-pointing triangle to see 
    the file contents. 
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/>

**Manifest Stuff**

<pre>
-----------------------------------------------------------------
manifest: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\manifest.tsv
rows: 2

000001 |  | literal | text/markdown | details-summary-markdown-regression | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000001_1782738768886_2026-06-29T091248886-0400_details-summary-markdown-regression.md
000002 |  | literal | text/html | details-summary-html-control | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000002_1782738776121_2026-06-29T091256121-0400_details-summary-html-control.html
-----------------------------------------------------------------

That's the previous tries and the start of this one...
Unless, as is the case now, this is a different machine,
so there are only the two new entries.
</pre>

<br/>

**Artifact Stuff** (Be sure to escape)

<pre>
########################################################################

-----------------------------------------------------------------
artifact_sequence: 000001
capture_sequence: 
role: literal
label: details-summary-markdown-regression
mime: text/markdown
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000001_1782738768886_2026-06-29T091248886-0400_details-summary-markdown-regression.md
exists: True

▼ Click the arrow to reveal logged Markdown text
This content was logged as text/markdown.

The regression question is whether exported timelines preserve this as renderable Markdown/HTML or incorrectly turn it into escaped or fenced source text.


HTML comments above.
Hopefully you can see the utility of jupy_log as a note-taking device.

-----------------------------------------------------------------
======================================================================

======================================================================
-----------------------------------------------------------------
artifact_sequence: 000002
capture_sequence: 
role: literal
label: details-summary-html-control
mime: text/html
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000002_1782738776121_2026-06-29T091256121-0400_details-summary-html-control.html
exists: True

▼ Click the arrow to reveal logged HTML text
This content was logged as text/html. It should be inserted as HTML in the generated HTML timeline.

## 10. Code cell 09 — log an HTML control version
HTML comments above.
Hopefully you can see the utility of jupy_log as a note-taking device.
-----------------------------------------------------------------


########################################################################



####################################################################
### ONLY FOR THIS REGRESSION TEST, WE dpypd.print_file TWO FILES ###
####################################################################



###############################################################
-----------------------------------------------------------

___File contents for___
   'jupy_log/artifacts/
000001_1782738768886_2026-06-29T091248886-0400_
details-summary-markdown-regression.md'___
-----------------------------------------------------------

&lt;details&gt;
&lt;summary&gt;Click the arrow to reveal logged Markdown text&lt;/summary&gt;

This content was logged as `text/markdown`.

The regression question is whether exported timelines preserve this as
renderable Markdown/HTML or incorrectly turn it into escaped or fenced
source text.

&lt;/details&gt;

&lt;!-- ## 9. Code cell 08 — log the original Markdown regression case --&gt;
&lt;!--   If I had left the python comment without the HTML comment,   --&gt;
&lt;!-- + it would have rendered as an `h2`. But you might only see    --&gt;
&lt;!-- + this in the dpypd.print_file version.                        --&gt;

&lt;br/&gt;HTML comments above.&lt;br/&gt;Hopefully you can see the utility of jupy_log
as a note-taking device.
-----------------------------------------------------------
===========================================================

===========================================================
-----------------------------------------------------------

___File contents for___
   'jupy_log/artifacts/
000002_1782738776121_2026-06-29T091256121-0400_
details-summary-html-control.html'___
-----------------------------------------------------------

&lt;details&gt;
&lt;summary&gt;Click the arrow to reveal logged HTML text&lt;/summary&gt;

&lt;p&gt;
This content was logged as &lt;code&gt;text/html&lt;/code&gt;. It should be inserted
as HTML in the generated HTML timeline.
&lt;/p&gt;

&lt;/details&gt;

## 10. Code cell 09 — log an HTML control version

&lt;!--   I can leave the python comment above as-is, since this is --&gt;
&lt;!-- + HTML and not markdown. But you might only see this in the --&gt;
&lt;!-- + dpypd.print_file version.                                 --&gt;

&lt;br/&gt;HTML comments above.&lt;br/&gt;Hopefully you can see the utility of jupy_log
as a note-taking device.
-----------------------------------------------------------


###############################################################
</pre>

</details>

---

For **11** (`Code cell 14 — basic ``%%jupy_capture`` smoke test`)

Expected capture roles include:

```text
input
stdout
display
```

The final expression `5` may be represented through one or more MIME
items, commonly including `text/plain`.

In [ ]:
%%jupy_capture --label capture-basic
message = "This stdout should be displayed and logged."
print(message)

## 11. Code cell 14 — basic `%%jupy_capture` smoke test

In [ ]:
## Code cell 15

dpypd.tree(f"{repo_root}/jupy_log")

In [ ]:
## Code cell 16 — Manifest Check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 17, ANYTHING NEW?

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(3)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(4)
display_banner()
print(); print()
display_banner("#", 72)

----

**New stuff after Code cell 17**

<details>
<summary>Click the right-pointing triangle to see 
    the file contents. 
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/><br/>

**Manifest Stuff**

<pre>
-----------------------------------------------------------------
manifest: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\manifest.tsv
rows: 4

000001 |  | literal | text/markdown | details-summary-markdown-regression | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000001_1782738768886_2026-06-29T091248886-0400_details-summary-markdown-regression.md
000002 |  | literal | text/html | details-summary-html-control | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000002_1782738776121_2026-06-29T091256121-0400_details-summary-html-control.html
000003 | 000001 | input | text/x-python | capture-basic-input | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000003_1782739051962_2026-06-29T091731962-0400_capture-basic-input.py
000004 | 000001 | stdout | text/plain | capture-basic-stdout | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000004_1782739052272_2026-06-29T091732272-0400_capture-basic-stdout.txt
-----------------------------------------------------------------
</pre>

<br/><br/>

**Artifact Stuff**

<pre>
########################################################################

-----------------------------------------------------------------
artifact_sequence: 000003
capture_sequence: 000001
role: input
label: capture-basic-input
mime: text/x-python
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000003_1782739051962_2026-06-29T091731962-0400_capture-basic-input.py
exists: True

message = "This stdout should be displayed and logged."
print(message)

## 11. Code cell 14 — basic `%%jupy_capture` smoke test

-----------------------------------------------------------------
======================================================================

======================================================================
-----------------------------------------------------------------
artifact_sequence: 000004
capture_sequence: 000001
role: stdout
label: capture-basic-stdout
mime: text/plain
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000004_1782739052272_2026-06-29T091732272-0400_capture-basic-stdout.txt
exists: True

This stdout should be displayed and logged.

-----------------------------------------------------------------


########################################################################
</pre>

</details>

<hr/>

For **12** (`Code cell 18 — basic ``%%jupy_tee`` alias test`)

Not really anything to comment about, but here it comes.

In [ ]:
%%jupy_tee --label tee-basic
tee_message = "The %%jupy_tee alias executed this cell."
print(tee_message)

tee_message.upper()

## 12. Code cell 18 — basic `%%jupy_tee` alias test

In [ ]:
## Code cell 19

dpypd.tree(f"{repo_root}/jupy_log")

In [ ]:
## Code cell 20 — Manifest Check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 21, ANYTHING NEW?

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(5)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(6)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(7)
display_banner()
print(); print()
display_banner("#", 72)

---

**New stuff after Code cell 21**

<details>
<summary>Click the right-pointing triangle to see 
    the file contents. 
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/><br/>

**Manifest Stuff**

<pre>
-----------------------------------------------------------------
manifest: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\manifest.tsv
rows: 7

000001 |  | literal | text/markdown | details-summary-markdown-regression | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000001_1782738768886_2026-06-29T091248886-0400_details-summary-markdown-regression.md
000002 |  | literal | text/html | details-summary-html-control | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000002_1782738776121_2026-06-29T091256121-0400_details-summary-html-control.html
000003 | 000001 | input | text/x-python | capture-basic-input | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000003_1782739051962_2026-06-29T091731962-0400_capture-basic-input.py
000004 | 000001 | stdout | text/plain | capture-basic-stdout | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000004_1782739052272_2026-06-29T091732272-0400_capture-basic-stdout.txt
000005 | 000002 | input | text/x-python | tee-basic-input | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000005_1782739120904_2026-06-29T091840904-0400_tee-basic-input.py
000006 | 000002 | stdout | text/plain | tee-basic-stdout | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000006_1782739120920_2026-06-29T091840920-0400_tee-basic-stdout.txt
000007 | 000002 | display | text/plain | tee-basic-display-01-01-text | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000007_1782739120931_2026-06-29T091840931-0400_tee-basic-display-01-01-text.txt
-----------------------------------------------------------------
</pre>

<br/><br/>

**Artifact Stuff**

<pre>
########################################################################

-----------------------------------------------------------------
artifact_sequence: 000005
capture_sequence: 000002
role: input
label: tee-basic-input
mime: text/x-python
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000005_1782739120904_2026-06-29T091840904-0400_tee-basic-input.py
exists: True

tee_message = "The %%jupy_tee alias executed this cell."
print(tee_message)

tee_message.upper()

## 12. Code cell 18 — basic `%%jupy_tee` alias test

-----------------------------------------------------------------
======================================================================

======================================================================
-----------------------------------------------------------------
artifact_sequence: 000006
capture_sequence: 000002
role: stdout
label: tee-basic-stdout
mime: text/plain
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000006_1782739120920_2026-06-29T091840920-0400_tee-basic-stdout.txt
exists: True

The %%jupy_tee alias executed this cell.

-----------------------------------------------------------------
======================================================================

======================================================================
-----------------------------------------------------------------
artifact_sequence: 000007
capture_sequence: 000002
role: display
label: tee-basic-display-01-01-text
mime: text/plain
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000007_1782739120931_2026-06-29T091840931-0400_tee-basic-display-01-01-text.txt
exists: True

'THE %%JUPY_TEE ALIAS EXECUTED THIS CELL.'
-----------------------------------------------------------------


########################################################################
</pre>

</details>

---

In [ ]:
%matplotlib inline

## 13.1. Code cell 22

_-^- Doing this early, because I want to. For **13.1**._

**13. Begin the notebook-state test**

**14. Description and preparation: A, B, C, then B, A**

The next five capture cells must be run in their displayed order first, then partially reversed:

```text
A first
B first
C
B second
A second
```

Expected results:

```text
A first  -> NameError because squared_numbers does not exist
B first  -> NameError because numbers does not exist
C        -> creates numbers successfully
B second -> creates squared_numbers successfully
A second -> computes the mean and creates the plot successfully
```

The reset cell immediately below, **14** (`Code cell 23 — reset hidden kernel state`), removes any old values that might make the deliberately incorrect first pass appear to work.

In [ ]:
%%jupy_capture --label abc-state-reset
state_names = [
    "numbers",
    "squared_numbers",
    "avg_value",
    "fig",
    "ax",
    "plot_path",
]

removed_names = []

for state_name in state_names:
    if state_name in globals():
        globals().pop(state_name)
        removed_names.append(state_name)
    ##endof:  if state_name in globals()
##endof:  for state_name in state_names

print("Removed prior state:", removed_names)
print("A and B should now fail on their first executions.")

## 14. Code cell 23 — reset hidden kernel state

In [ ]:
## Code cell 24

dpypd.tree(f"{repo_root}/jupy_log")

In [ ]:
##  Code cell 25 — Manifest Check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 26, ANYTHING NEW?

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(8)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(9)
display_banner()
print(); print()
display_banner("#", 72)

---

**New stuff after Code cell 26**

<details>
<summary>Click the right-pointing triangle to see 
    the file contents. 
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/><br/>

**Manifest Stuff**

<pre>
-----------------------------------------------------------------
manifest: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\manifest.tsv
rows: 9

000001 |  | literal | text/markdown | details-summary-markdown-regression | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000001_1782738768886_2026-06-29T091248886-0400_details-summary-markdown-regression.md
000002 |  | literal | text/html | details-summary-html-control | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000002_1782738776121_2026-06-29T091256121-0400_details-summary-html-control.html
000003 | 000001 | input | text/x-python | capture-basic-input | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000003_1782739051962_2026-06-29T091731962-0400_capture-basic-input.py
000004 | 000001 | stdout | text/plain | capture-basic-stdout | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000004_1782739052272_2026-06-29T091732272-0400_capture-basic-stdout.txt
000005 | 000002 | input | text/x-python | tee-basic-input | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000005_1782739120904_2026-06-29T091840904-0400_tee-basic-input.py
000006 | 000002 | stdout | text/plain | tee-basic-stdout | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000006_1782739120920_2026-06-29T091840920-0400_tee-basic-stdout.txt
000007 | 000002 | display | text/plain | tee-basic-display-01-01-text | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000007_1782739120931_2026-06-29T091840931-0400_tee-basic-display-01-01-text.txt
000008 | 000003 | input | text/x-python | abc-state-reset-input | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000008_1782739193744_2026-06-29T091953744-0400_abc-state-reset-input.py
000009 | 000003 | stdout | text/plain | abc-state-reset-stdout | D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000009_1782739193761_2026-06-29T091953761-0400_abc-state-reset-stdout.txt
-----------------------------------------------------------------
</pre>

<br/><br/>

**Artifact Stuff**

<pre>
########################################################################

-----------------------------------------------------------------
artifact_sequence: 000008
capture_sequence: 000003
role: input
label: abc-state-reset-input
mime: text/x-python
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000008_1782739193744_2026-06-29T091953744-0400_abc-state-reset-input.py
exists: True

state_names = [
    "numbers",
    "squared_numbers",
    "avg_value",
    "fig",
    "ax",
    "plot_path",
]

removed_names = []

for state_name in state_names:
    if state_name in globals():
        globals().pop(state_name)
        removed_names.append(state_name)
    ##endof:  if state_name in globals()
##endof:  for state_name in state_names

print("Removed prior state:", removed_names)
print("A and B should now fail on their first executions.")

## 14. Code cell 23 — reset hidden kernel state

-----------------------------------------------------------------
======================================================================

======================================================================
-----------------------------------------------------------------
artifact_sequence: 000009
capture_sequence: 000003
role: stdout
label: abc-state-reset-stdout
mime: text/plain
path: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\000009_1782739193761_2026-06-29T091953761-0400_abc-state-reset-stdout.txt
exists: True

Removed prior state: []
A and B should now fail on their first executions.

-----------------------------------------------------------------


########################################################################
</pre>

</details>

<hr/>

**`=============`**<br/>
**`Cell A, below`**<br/>
**`=============`**<br/>
- For **15**
  (`Code cell         27 — A, 1st execution: expected failure`).
  This output is for cells **below** this Markdown cell.

  - Expected result:

```text
NameError involving squared_numbers
```

  - The capture should still log:
    * the input
    * the exception
    * any output produced before the exception, if present
  - _All this output comes from Code cells **below** this Markdown cell._
  - ACTUAL OUTPUT:

```text
---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[28], line 7
      3 ##+ will do it before we come back for the successful run of
      4 ##+ already did. Same for numpy.
      5 ##+ This is intentional notebook-state dependence for the A/B/C test.
      6 
----> 7 avg_value = squared_numbers.mean()
      8 
      9 fig, ax = plt.subplots(figsize=(8, 4))
     10 

NameError: name 'squared_numbers' is not defined
[DONE] Capture 000004: logged 2 artifact(s).
[MMJL] %%jupy_capture captured an execution error.
```

  - (Output again)
    - This time, with exception text (in the Jupyter notebook,
      the text with a red background) bolded (immediately
      below.) The dots anchoring and aligning the right side 
      weren't part of original output, nor of any following 
      output in which they appear.

**`---------------------------------------------------------------------------            .`**<br/>
**`NameError                                 Traceback (most recent call last)            .`**<br/>
**`Cell In[28], line 7                                                                    .`**<br/>
**`      3 ##+ will do it before we come back for the successful run of                   .`**<br/>
**`      4 ##+ already did. Same for numpy.                                               .`**<br/>
**`      5 ##+ This is intentional notebook-state dependence for the A/B/C test.          .`**<br/>
**`      6                                                                                .`**<br/>
**`----> 7 avg_value = squared_numbers.mean()                                             .`**<br/>
**`      8                                                                                .`**<br/>
**`      9 fig, ax = plt.subplots(figsize=(8, 4))                                         .`**<br/>
**`     10                                                                                .`**<br/>
**`                                                                                      ..`**<br/>
**`NameError: name 'squared_numbers' is not defined                                       .`**<br/>
`[DONE] Capture 000004: logged 2 artifact(s).                                           .`<br/>
**`[MMJL] %%jupy_capture captured an execution error.                                     .`**

  - Output of `dpypd.tree()`

```text
+ D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log
    + artifacts/
        + 000001_1782738768886_2026-06-29T091248886-0400_details-summary-markdown-regression.md
        + 000002_1782738776121_2026-06-29T091256121-0400_details-summary-html-control.html
        + 000003_1782739051962_2026-06-29T091731962-0400_capture-basic-input.py
        + 000004_1782739052272_2026-06-29T091732272-0400_capture-basic-stdout.txt
        + 000005_1782739120904_2026-06-29T091840904-0400_tee-basic-input.py
        + 000006_1782739120920_2026-06-29T091840920-0400_tee-basic-stdout.txt
        + 000007_1782739120931_2026-06-29T091840931-0400_tee-basic-display-01-01-text.txt
        + 000008_1782739193744_2026-06-29T091953744-0400_abc-state-reset-input.py
        + 000009_1782739193761_2026-06-29T091953761-0400_abc-state-reset-stdout.txt
        + 000010_1782739271877_2026-06-29T092111877-0400_abc-first-pass-a-input.py
        + 000011_1782739272516_2026-06-29T092112516-0400_abc-first-pass-a-exception.txt
    + manifest.tsv
    + staging/
    + timelines/
```

  - **All** output concerning `manifest.tsv`

```text
bar manifest
```

  - Content of `_abc-first-pass-a-input.py` input text (code)

```python
foo.py
```

  - Content of `_abc-first-pass-a-exception.txt` output text (exception)

```text
foo txt
```

Take note that the code file matches the code cell exactly,

<span style="font-size:175%">but, <strong>kamMA</strong>, the
exception text shown on the screen in the Jupyter
notebook <strong>does not</strong> match the text in the
exception logging file.</span>

<span style="font-size:125%">The vital information matches,
though. This section will probably be expanded with a
_What I learned about Jupyter_ section.</span>

`---`

**What I learned about Jupyter from Cell A's failure**

---

**` Cell A     `**<br/>
**` Second try `**

- For **20**
  (`Code cell (still) 27 — A, 2nd execution: expected success and plot capture`)
- But, as far as execution number, this would be cell 39+x=44 (x=5),
  so I'm going to call it ~~**27/39+x**~~ **27/44**
  - The only difference in the input is the change,
    - from:
      - `%%jupy_capture --label abc-first-pass-a`
    - to:
      - `%%jupy_capture --label abc-second-pass-a`
  
  - ACTUAL OUTPUT
 
```text
saved explicit plot file: D:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\staging\abc_backwards_right.png
explicit plot exists: True


[``````````````````````````````````````````````]
[{ The actual plot showed up here; see below. }]
[______________________________________________]


[DONE] Capture 000010: logged 4 artifact(s).
```

The rest of the successful run's files' contents can be seen in the output
of Code cells ~~**39+x+3** (30|**39+x+3**) and **39+x+4** (31|**39+4**)~~
**46** (30|**46**) and **47** (31|**47**)

In [ ]:
%%jupy_capture --label abc-first-pass-a
##done before# import matplotlib.plotly as plt
##  P.S. Cell A does not import matplotlib here, because Cell C
##+ will do it before we come back for the successful run of
##+ already did. Same for numpy.
##+ This is intentional notebook-state dependence for the A/B/C test.

avg_value = squared_numbers.mean()

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    numbers,
    squared_numbers,
    marker="o",
)

ax.axhline(
    avg_value,
    linestyle="--",
    label=f"mean squared value = {avg_value:.2f}",
)

ax.set_title(
    "Forwards-wrong / backwards-right notebook-state test"
)
ax.set_xlabel("number")
ax.set_ylabel("number squared")
ax.grid(True)
ax.legend()

plot_path = (
    repo_root
    / "jupy_log"
    / "staging"
    / "abc_backwards_right.png"
)

plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print("saved explicit plot file:", plot_path)
print("explicit plot exists:", plot_path.exists())

plt.show()

## 15. Code cell         27 — A, 1st execution: expected failure
#
## and
#
## 20. Code cell (still) 27 — A, 2nd execution: expected success and plot capture
##     call it Code Cell 27|43

In [ ]:
## Code cell 28|44

dpypd.tree(f"{repo_root}/jupy_log")

<br/><hr/>
<div>
<span style="font-size:150%; font-face:bold">MUST 
    MANUALLY PUT IN THE THE FILENAMES, BELOW!</span>
</div>
<div>Also make sure you have the 
    correct <code>first-pass</code> vs.
    the <code>second-pass</code> part
    of the filename.</div>

In [ ]:
# Code cell 29|45

print("JUST BECAUSE THIS WAS GOOD FOR DEBUGGING, I'M")
print("GOING TO FIND ANYTHING FROM THE PREVIOUS TRY")
print("AND FROM THIS TRY THAT HAVE TO DO WITH CELL A.")

import contextlib
import io

import filecmp

buffer = io.StringIO()  # string streamer

with contextlib.redirect_stdout(buffer):
   dpypd.tree(f"{repo_root}/jupy_log")
##endof:  with contextlib.redirect_stdout(buffer)

tree_out_str = buffer.getvalue()

grep_a_cell_results = [
    line
    for line in tree_out_str.splitlines()
    if r"abc-" in line and r"pass-a-" in line
]

print(*grep_a_cell_results, sep="\n")
print()
print()
display_banner()
display_banner("=")
display_banner("#")
print()
display_banner("#")
display_banner("=")
display_banner()
print()

filename_1 = (
    repo_root / "jupy_log" / "artifacts" 
              / ("000010"
                 "details-summary-m
#                "abc-first-pass-a-input.py")
)
filename_2 = (
    repo_root / "jupy_log" / "artifacts" 
              / ("000011"
                 "abc-first-pass-a-exception.txt")
)

#  Set shallow=False to compare internal contents rather than 
#+ just file metadata
are_identical = filecmp.cmp(filename_1, filename_2, shallow=False)

correct_verbiage = (
    "  have contents that are identical."
    if are_identical
    else
    "  have contents that are not identical."
)

print(f"filename_1: '{filename_1}'\n  and\nfilename_2: '{filename_2}'\n{correct_verbiage}")

**Quick debug-type results from code-cell 30**

```text
foo
```

<br/><hr/><br/>

**Quick debug-type results from Code cell 45**

```text
bar
```

In [ ]:
##  Code cell 30|46 — Manifest Check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 31|47, ANYTHING NEW?

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(22)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(23)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(24)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(25)
display_banner()
print()
print("I'm not going to output the binary image file. The")
print("file directly above should let you know it exists.")
print("(Hint: not on the first pass.)")
print()
print("But the image will have shown up on the second pass.")
print(); print()
display_banner("#", 72)

---

**New stuff after Code cell (30 &amp;) 31, 
later for (46 &amp;) 47**

_Those all relate to **Cell A**_


**` Cell A     .`**<br/>
**` First try  .`**

<details>
<summary>Click the arrow to see the file contents for 
    Code cells 30 &amp; 31.
    Both related to <strong>Cell A</strong>.
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/><hr/><br/>
**For Code cell 30 — Cell A related**
<br/>
**Manifest Stuff**

<pre>
-----------------------------------------------------------------
blah
-----------------------------------------------------------------
</pre>

<br/><hr/><br/>
**For code cell 31 — Cell A related**
<br/>
**Artifact Stuff** (Be sure to escape)

<pre>
########################################################################

-----------------------------------------------------------------

I'm not going to output the binary image file. The
file directly above should let you know it exists.
(Hint: not on the first pass.)


########################################################################
</pre>

</details>

<br/><hr/><br/>

**` Cell A     .`**<br/>
**` Second try .`**

<details>
<summary>Click the arrow to see the file contents for 
    Code cell <strike>28|<strong>39+x</strong> 
    28|<strong>43</strong> &mdash; THE <strong>Cell
    A</strong> second attempt.
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

**Note for Cell A (second), Code cell ~~39+x~~ 43**

_These observations are for the first time I ran the cell
and will probably not match the plot in this notebook._ -DWB

<div style="background-color: cyan; max-width: 750px; width: fit-content;">
<span style="font-family: monospace;">
<strong><em>
That plot has a very interesting way of showing that one degenerate<br/>
(repeated) <strike>, degenerate might not be the right word, eh 
kamMA?)</strike> value is 64<br/>
in <code style="background-color: cyan;">squared_numbers</code>, seen 
in the plot on the x-axis as 8 
in <code style="background-color: cyan;">numbers</code>.<br/>
One can see that more than two lines <em>come in</em>/<em>go out</em> at
(8, 64).<br/>
(9, 81) is another repeated value pair. The reason we don&apos;t 
see<br/>
the extra visits is simple.
</em></strong>
</span>

<div style="background-color: white"><br></div>

<details>
<summary>
<span style="font-family: monospace;">
<strong><em>Click to see the reason, 
if nested details/summary tags are allowed.</em></strong>
</span>
</summary>

<div style="background-color: white;">They are allowed!</div>
    

<div style="background-color: #FF8B00;">
The <code style="background-color: #FF8B00;">numbers</code> array is 
represented 
as <code style="background-color: #FF8B00;"
       >[8  2  8  6  8  9  9  5  3  10]</code>.

First of all, 9 goes from itself to itself in a connected scatter plot
(not the best choice of visualization in this situation, we will have
 to discuss that, kamMA.) The scatter plot makes things interesting in
another way; If we trace traversals between numbers, we can see that
several paths are traversed mutliple times. Let's walk the whole...
graph-like thingie, then order each path from small number to big,
then group our edge-like thingies.

<code style="background-color: #FF8B00;"
    >(8 → 2, 2 → 8, 8 → 6, 6 → 8, 8 → 9, 9 → 9, 9 → 5, 5 → 3, 3 → 10)</code>

each directed segment (-ish) becomes an edge (-ish) when one goes
<code style="background-color: #FF8B00;"
    >smaller ↔ same-or-larger</code> in every case.

<code style="background-color: #FF8B00;"
    >(2 ↔ 8, 2 ↔ 8, 6 ↔ 8, 6 ↔ 8, 8 ↔ 9, 9 ↔ 9, 5 ↔ 9, 3 ↔ 5, 3 ↔ 10)</code>

Now, we can order and group these, comparing firsts, and if the firsts
are the same, comparing seconds.

<code style="background-color: #FF8B00;"
    >(
  2↔8, 2↔8,  # Edge traversed twice
  3↔5,
  3↔10,
  5↔9,
  6↔8, 6↔8,  # Edge traversed twice
  8↔9,
  9↔9
)</code>

So we have two edges traversed twice. What an unnecessary but
interesting tangent. I was just trying to figure out how I could
analyze the distribution.
</div>
</div>

</details>

<br/>

<span style="font-family: monospace; background-color: cyan;"
    ><strong><em>Cyan for fun!!!</em></strong></span>

</details>

<br/><hr/><br/>

<details>
<summary>Click the arrow to see the file contents for 
    Code cells <strike>39+x+3 &amp; 
    39+x+4.</strike> 46 &amp; 47.
    All related to <strong>Cell A</strong>.
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>
</summary>
    
**For Code cell ~~39+x+3~~ 46 — Cell A related** 
<br/>
**Manifest Stuff**

<pre>
-----------------------------------------------------------------
blah
-----------------------------------------------------------------
</pre>

<br/><hr/><br/>
**For Code cell ~~39+x+4~~ 47 — Cell A related**
<br/>
**Artifact Stuff** (Be sure to escape)

<pre>
########################################################################
blah
########################################################################
</pre>

</details>

<hr/>

---

<span style="font-size: 150%; font-face: bold;">You 
can skip the next stuff (to the crossed out 
<code>h2</code>, the <strong>## 19</strong>,) where
your first code cell will be Code cell 48, 
<strong>if you've done B and A twice</strong>. 
Otherwise, keep going down with 
Cell B, which is Code cell 32.</span>

---

---

**If you have finished A, B, C, then back to B, A**

**It's time to go to Code cell 39+x+4+1=48**, a.k.a. 
Code cell Z, where $Z=48$ (because $x=4$) which is 
is after **\## 19**, the crossed out `h2`.

Links never seem to work. Scrolling time.

Unless you haven't done A twice, in which case you
can just go on to the next cell.

---

---

**`=============`**<br/>
**`Cell B, below`**<br/>
**`=============`**<br/>
- For **16**
  (`Code cell         32 — first execution: expected failure`).
  This output is for cells **below** this Markdown cell.

  - Expected result:

```text
NameError involving squared_numbers
```

  - The capture should still log:
    * the input
    * the exception
    * any output produced before the exception, if present
  - _All this output comes from Code cells **below** this Markdown cell._
  - ACTUAL OUTPUT:

```text
blah
```

  - (Output again)
    - This time, with exception text (in the Jupyter notebook,
      the text with a red background) bolded (immediately
      below.) The dots anchoring and aligning the right side 
      weren't part of original output, nor of any following 
      output in which they appear.

**`---------------------------------------------------------------------------            .`**<br/>
**`NameError                                 Traceback (most recent call last)            .`**<br/>
**`Cell In[34], line 1                                                                    .`**<br/>
**`----> 1 squared_numbers = numbers ** 2                                                 .`**<br/>
**`      2                                                                                .`**<br/>
**`      3 print("squared_numbers:", squared_numbers)                                     .`**<br/>
**`      4                                                                                .`**<br/>
**`                                                                                       .`**<br/>
**`NameError: name 'numbers' is not defined                                               .`**<br/>
`[DONE] Capture 000006: logged 2 artifact(s).                                           .`<br/>
**`[MMJL] %%jupy_capture captured an execution error.                                     .`**

  - Output of `dpypd.tree()`

```text
bar tree
```

  - All content of `manifest.tsv`

```text
bar manifest
```

  - Content of `_abc-first-pass-b-input.py` input text (code)

```python
bar py
```

  - Content of `_abc-first-pass-b-exception.txt` output text (exception)

```text
bar tt
```

<br/>

Take note that the code file matches the code cell exactly,

<span style="font-size:175%">but, <strong>kamMA</strong>, just
as in the first, failed Cell A, the
exception text shown on the screen in the Jupyter
notebook <strong>does not</strong> match the text in the
exception logging file.</span>

<span style="font-size:125%">Interestingly, though the vital 
information matches, there is an interesting (but ultimately
unhelpful) hint that
appears in the logged exception versus the on-screen
exception. Once again, this section will probably be 
expanded with a _What I learned about Jupyter_ 
section.</span>

`---`

**What I learned about Jupyter from Cell B's failure**

---

**` Cell B     `**
**` Second try `**

- For **18** (`Code cell (still) 32 — second execution: expected success`)
- This would be code cell 32+y=40, (y=8) so I'm going to call it ~~**32|32+y**~~ **32|40**
  - The only difference in the input is the change,
    - from:
      - `%%jupy_capture --label abc-first-pass-b`
    - to:
      - `%%jupy_capture --label abc-second-pass-b`
  
  - ACTUAL OUTPUT
 
```text
blah
```

The rest of the successful run's files' contents can be seen in the output of Code cells ~~**32+y+2** (with 34|**32+y+2**) and **32+y+3** (with 35|**32+y+3**)~~ **42** (with 34|**42**) and **43** (with 35|**43**)

In [ ]:
%%jupy_capture --label abc-first-pass-b
squared_numbers = numbers ** 2

print("squared_numbers:", squared_numbers)

## 16. Code cell         32 — B, first execution: expected failure
#
## and
#
## 18. Code cell (still) 32 — B, second execution: expected success
##     call it Code cell 32|40

In [ ]:
## Code cell 33|41

dpypd.tree(f"{repo_root}/jupy_log")

In [ ]:
##  Code cell 34|42 — Manifest Check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 35|43, ANYTHING NEW?

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(20)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(21)
display_banner()
print(); print()
display_banner("#", 72)

---

**New stuff after Code cell (34 &amp; ) 35, later for (43 &amp;) 44**

_Those all relate to **Cell B**_

<details>

<summary>Click the arrow to see the file contents for 
    Code cells (35 &amp; 36) then <strike>( 33+y+2 &amp; 33+y+3
    )</strike>. 43 &amp; 44
    All related to <strong>Cell B</strong>.
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/><hr/><br/>
**For Code cell 34 — Cell B related**
<br/>
**Manifest Stuff**

<pre>
-----------------------------------------------------------------
blah
-----------------------------------------------------------------
</pre>

<br/><hr/><br/>
**For Code cell 35 — Cell B related**
<br/>
**Artifact Stuff**

<pre>
########################################################################
blah
########################################################################
</pre>

<br/><hr/><br/>

**` Cell B     `**
**` Second try `**

**For Code cell ~~33+y+2~~ 43 — Cell B related**
<br/>
**Manifest Stuff**

<pre>
-----------------------------------------------------------------
blah
-----------------------------------------------------------------
</pre>

<br/><hr/><br/>
**For Code cell ~~33+x+3~~ 44 — Cell B related**

**Artifact Stuff**

<pre>
########################################################################
blah
########################################################################
</pre>

</details>

<hr/>

---

<span style="font-size: 150%; font-face: bold;">If this has been your 
second time doing B, go back to A (Code cell 
28|<strong>45</strong>).</span>

And while you're having a look at Code cell (28|)**45**, don't 
forget to look at the Markdown above, which
details the change you make to the value handed the
<code>--label</code> option, and

<span style="font-size:150%">that change is to go from
the old<br/><code>--label abc-first-pass-a</code><br/>to the 
new<br/><code>--label abc-second-pass-a</code></span>,

Then, after finishing B and the following cells, you'll go
to A again (you'll be told where to go after finishing B.)

<span style="font-size:150%">If it is your first time doing 
the **Cell B** stuff, go ahead and continue down.</span>

---

**`=============`**<br/>
**`Cell C, below`**<br/>
**`=============`**<br/>
- For **17** (`Code cell 36 — C: expected success`)

  - Expected result:
    - Success. 
  - File contents for successful run
    - Look at the output of **Code cells 38-39**
  - ACTUAL OUTPUT

```text
foo
```

In [ ]:
%%jupy_capture --label abc-cell-c
import numpy as np
import matplotlib.pyplot as plt

## Older global-random-state style, like MLU
# numbers = np.random.randint(1, 11, size=10)

rng = np.random.default_rng()
numbers = rng.integers(1, 11, size=10)

print("numbers:", numbers)

## 17. Code cell 36 — C: expected success

In [ ]:
## Code cell 37

dpypd.tree("f{repo_root}/jupy_log")

In [ ]:
##  Code cell 38 — Manifest check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 39 — ANYTHING NEW?

display_banner("#", 72)
print()
display_banner()
show_artifact(16)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(17)
display_banner()
print(); print()
display_banner("#", 72)

---

**New stuff after Code cell 39**

<details>
<summary>Click the right-pointing triangle to see 
    the file contents. 
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

<br/><hr/><br/>
**For Code cell 38 (Manifest check) — Cell C related**

<pre>
-----------------------------------------------------------------
blah
-----------------------------------------------------------------
</pre>

<br/><hr/><br/>
**For Code cell 39 (Looking at artifacts) — Cell C related**

<pre>
########################################################################
blah
########################################################################
</pre>

</details>

---

<span style="font-size: 150%; font-face: bold;">If you 
haven't done B (and A) twice, go up and do B (Code cell 
32|<strong>40</strong>.)</span>

And while you're having a look at Code cell 32|**40**, 
don't forget to look at the Markdown above, which
details the change you make to the value handed the
<code>--label</code> option, and

<span style="font-size:150%">that change is to go from
the old<br/><code>--label abc-first-pass-b</code><br/>to the 
new<br/><code>--label abc-second-pass-b</code></span>,

Then, after finishing B and the following cells, you'll go
to A again (you'll be told where to go after finishing B.)

---

---

(**18** combined with **16** — both are Cell B) See above

---



## 19.~~Code cell~~ — ~~configure Matplotlib inline rendering~~ 

#### Note: trying (later: tried and succeeded) to do this earlier, in **13.1**

~~Run this before the successful A cell.~~

~~\`\`\``python`~~<br/>
~~`%matplotlib inline`~~<br/>
~~\`\`\`~~

---

---

(**20** combined with **16** — both are Cell A) See above

---

---

<span style="font-size: 150%; font-face: bold;">If you 
have done the whole A, B, C, then back for B and A, go ahead and
continue. Code cell Z will be your first to do, now. The last
one you should have completed was <strike>(31|)**39+x+4**</strike> 
but x=4, so the last one should have been 31|**47**, and 
40+x+4=Z-1, so we go to Code cell Z=48 </span>

---

---

_For **15**&ndash;**20**, looking towards **21**_

This tests two related but distinct things:

* inline PNG capture by `%%jupy_capture`
* creation of a file that `%jupy_file` can log explicitly

In [ ]:
## Code cell Z = 40+x+5 | _{x=4} => Code cell 48

dpypd.tree(f"{repo_root}/jupy_log")

In [ ]:
##  Code cell 49 — Manifest check

display_banner()
show_manifest(limit=20)
display_banner()

<span style="font-size:150%; font-face:bold">MUST CHANGE (or at least
    check) <code>show_artifact</code> NUMBERS BELOW!</span>

In [ ]:
##  Code cell 50, ~~ANYTHING NEW?~~ Nope, a review

########################################
##     REMEMBER TO MATCH NUMBERS!     ##
########################################

display_banner("#", 72)
print()
display_banner()
show_artifact(22)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(23)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(24)
display_banner()
display_banner("=", 70)
print()
display_banner("=", 70)
display_banner()
show_artifact(25)
display_banner()
print(); print()
display_banner("#", 72)

---

**~~New~~ Review stuff after Code cell 50**

<details>
<summary>Click the right-pointing triangle to see 
    the file contents. 
    Note that any internal HTML should be escaped,
    and downward-pointing triangles should come
    before expanded <code>&lt;detail&gt;</code>,
    <code>&lt;summary&gt;</code> pairs.</summary>

For the **manifest** (Code cell 49)

<pre>
-----------------------------------------------------------------
blah
-----------------------------------------------------------------
</pre>

---

For the **artifact(s)** (Code cell 50)

<pre>
blah
</pre>

</details>

<hr/>

For **21** (`Code cell 51 — test ``%jupy_file`` with the saved plot`)

The magic currently appends `-01` to the supplied label because it
supports logging multiple paths in one call.

In [ ]:
get_ipython().run_line_magic(
    "jupy_file",
    (
        "--label abc-explicit-plot "
        "--mime image/png "
        f'"{plot_path}"'
    ),
)

## 21. Code cell 51 — test `%jupy_file` with the saved plot

In [ ]:
## 22. Code cell 52 — initial manifest inspection and validation

%jupy_inspect

print("\n" + "-" * 72 + "\n")

%jupy_validate

For **23** 
(`## Code cell `
~~Z+5~~ 
` 53 — instantiate a direct logger for programmatic inspection`)

This logger points at the same root as the registered magics.

In [ ]:
## 23. Code cell 53 — instantiate a direct logger for programmatic inspection

log_root = repo_root / "jupy_log"
logger = MultimodalJupyLogger(root=log_root)

manifest_rows = logger.read_manifest_rows()

print("log_root:", log_root)
print("manifest:", logger.manifest)
print("artifact directory:", logger.artifacts)
print("manifest rows:", len(manifest_rows))

For **24** (`## Code cell `
~~Z+6~~ 
` 54 — inspect the most recent manifest rows`)

Look for groups corresponding to:

```text
details-summary-markdown-regression
details-summary-html-control
capture-basic
tee-basic
abc-state-reset
abc-first-pass-a
abc-first-pass-b
abc-cell-c
abc-second-pass-b
abc-second-pass-a
abc-explicit-plot
```

In [ ]:
## 24. Code cell 54 — inspect the most recent manifest rows

catch_all_row_count = 30
all_rows = manifest_rows[-catch_all_row_count:]

for row in all_rows:
    print(
        row.get("artifact_sequence", ""),
        "capture=" + (row.get("capture_sequence", "") or "-"),
        "role=" + row.get("role", ""),
        "mime=" + row.get("mime", ""),
        "label=" + row.get("label", ""),
    )
##endof:  for row in recent_rows

In [ ]:
## 25. Code cell 55 — inspect the log tree with `dpypd.tree`

dpypd.tree(
    this_dir=log_root,
    dirs_to_exclude=[
        "__pycache__",
    ],
    files_to_exclude=[
        ".pyc",
    ],
)

In [ ]:
## 26. Code cell 56 — use the `wc -l` analog

manifest_line_count = dpypd.count_lines(
    logger.manifest,
    do_print=True,
)

print(
    "Expected manifest data rows:",
    manifest_line_count - 1,
)
print(
    "Rows returned by read_manifest_rows:",
    len(manifest_rows),
)
print(
    "Counts agree:",
    manifest_line_count - 1 == len(manifest_rows),
)

For **27** (`Code cell `
~~Z+9~~
` 57 — use the ``cat`` analog on the manifest`)

Verify visually that:

* `artifact_sequence` increases monotonically
* capture artifacts share a `capture_sequence`
* the first failed A and B have exception rows
* the later B and A have successful output rows
* filenames contain both the sequence and epoch milliseconds

In [ ]:
## Code cell 57 — use the `cat` analog on the manifest

dpypd.print_file(
    logger.manifest,
    show_line_numbers=True,
)

For **28** (`Code cell `
~~Z+10~~
58` — generate both timelines`)

The equivalent magic calls are:

```python
%jupy_markdown
%jupy_html
```

Using the direct logger here makes the returned paths easy to retain.

In [ ]:
## 28. Code cell 58 — generate both timelines

markdown_timeline_path = logger.build_markdown()
html_timeline_path = logger.build_html()

print("Markdown timeline:", markdown_timeline_path)
print("HTML timeline:", html_timeline_path)

In [ ]:
## 29. Code cell 59 — count timeline lines

n_markdown_lines = dpypd.count_lines(
    markdown_timeline_path,
    do_print=True,
)

n_html_lines = dpypd.count_lines(
    html_timeline_path,
    do_print=True,
)

For **30** (`Code cell `
~~Z+12~~
` 60 — print the Markdown timeline`)

Things to check:

* artifact and capture sequences appear
* roles appear
* failed A and B occur before successful C, B, and A
* Python input is preserved
* exception text is preserved
* image links are present
* the Markdown `<details>` regression item is probably fenced rather than rendered

---

---

---

<span style="font-size: 350%; font-face: bold;">BIG PRINT MARKDOWN</span>

---

In [ ]:
## 30. Code cell 60 — print the Markdown timeline

dpypd.print_file(
    markdown_timeline_path,
    show_line_numbers=False,
)

---

---

<span style="font-size: 500%; font-face: bold;">REALLY BIG PRINT HTML</span>

---

For **31** (`Code cell `
~~Z+13~~
` 61 — inspect the raw HTML timeline source`)

This may be lengthy, but it is useful for this first pass.

Search visually for:

```text
details-summary-markdown-regression
details-summary-html-control
abc-first-pass-a
abc-first-pass-b
abc-second-pass-a
image/png
```

In [ ]:
## 31. Code cell 61 — inspect the raw HTML timeline source

dpypd.print_file(
    html_timeline_path,
    show_line_numbers=True,
)

---

<span style="font-size: 250%; font-face: bold;">THAT WAS THE HTML CODE</span>

---

For **32** (`Code cell `
~~Z+14~~
` 62 — render the generated HTML timeline in the notebook`)

Regression expectations:

* the `text/html` control should show a clickable `<details>` arrow
* the `text/markdown` version may show escaped `<details>` source
* the successful A plot should appear if its captured `image/png` path is
  usable from the generated timeline
* exception entries should be visible in chronological artifact order

---

<span style="font-size: 350%; font-face: bold;">HERE COMES THE RENDERED HTML</span>

---

In [ ]:
## 32. Code cell 62 — render the generated HTML timeline in the notebook

from IPython.display import HTML, display

display(
    HTML(
        filename=str(html_timeline_path),
    )
)

No image display due to the way Jupyter's server can get and show path stuff.

In [ ]:
## 33. Code cell 63 — final validation after exports

missing_paths = logger.validate_manifest()

print("\nFinal status:")
print("  manifest rows:", len(logger.read_manifest_rows()))
print("  missing artifact paths:", len(missing_paths))
print("  Markdown timeline exists:", markdown_timeline_path.exists())
print("  HTML timeline exists:", html_timeline_path.exists())
print("  explicit plot source exists:", plot_path.exists())

In [ ]:
## 34. Code cell 64 — final ordering assertions

final_rows = logger.read_manifest_rows()

artifact_sequences = [
    int(row["artifact_sequence"])
    for row in final_rows
]

sequences_are_monotonic = (
    artifact_sequences
    == sorted(artifact_sequences)
)

sequences_are_unique = (
    len(artifact_sequences)
    == len(set(artifact_sequences))
)

filenames_have_sequence_prefixes = all(
    pathlib.Path(row["path"]).name.startswith(
        f"{int(row['artifact_sequence']):06d}_"
    )
    for row in final_rows
)

print("Artifact sequences monotonic:", sequences_are_monotonic)
print("Artifact sequences unique:", sequences_are_unique)
print(
    "Filenames begin with artifact sequence:",
    filenames_have_sequence_prefixes,
)

## 35. Markdown cell — record the result

### Test-result notes

Record the observations here:

- [x] repository-root discovery worked
- [x] imports worked
- [x] utility module alias worked
- [x] individual utility imports worked
- [x] magics registered
- [x] notebook-native `<details>` worked
- [x] `text/markdown` regression behavior observed
- [x] `text/html` control rendered
- [x] `%%jupy_capture` executed and logged
- [x] `%%jupy_tee` executed and logged
- [x] first A failed as expected
- [x] first B failed as expected
- [x] C succeeded
- [x] second B succeeded
- [x] second A succeeded
- [x] inline Matplotlib plot was captured
- [x] explicit `%jupy_file` plot was logged
- [x] manifest sequences were monotonic
- [x] capture groups were visible
- [x] filenames contained millisecond timestamps
- [x] validation reported no missing artifacts
- [x] Markdown timeline was generated
- [x] HTML timeline was generated
- [x] timeline execution history was reconstructable

<span style="font-size:250%;">Go down to the very end, run the history, and try the PDF save.</span>

In [ ]:
%history

In [ ]:
import multimodal_jupy_logger.jupy_pdf_utils

%backup_pdf